# Workshop — IA Responsável na Prática · viés e *fairness*

**Público:** graduação em Ciência da Computação · **Formato:** 1 encontro de 2h · **Trabalho:** em grupos, um estudo de caso por grupo

| Parte | O que vamos fazer | Tempo |
|---|---|---|
| **1** | Treinar um modelo (direto ao ponto) | ~15 min |
| **2** | Onde este modelo pode ser injusto? | ~15 min |
| **3** | Auditoria de *fairness*: medir, tentar consertar, descobrir o *trade-off* | ~50 min |
| **4** | O dilema ético do seu caso | ~10 min |
| **5** | Painel final e apresentação relâmpago | ~15 min |

### Como usar este notebook
- Execute as células **em ordem** (Shift+Enter).
- Células marcadas com 🔧 **DECISÃO** pedem que o grupo **edite uma variável** antes de executar. Não existe resposta certa única — cada escolha tem consequências, e vocês vão vê-las.
- Células marcadas com ✏️ **TAREFA** trazem perguntas para discutir e anotar.
- **Ao chegar, execute já as duas primeiras células** (bibliotecas e infraestrutura) para ganhar tempo.

> **Preparação do ambiente** (uma única vez): `pip install pandas scikit-learn matplotlib` — ou abra no Google Colab, que já tem tudo.

In [ ]:
# Setup — apenas execute esta célula
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 4)
print("✅ Bibliotecas carregadas. Bom workshop!")

### ⚙️ Infraestrutura do workshop — apenas execute
A célula abaixo prepara os **5 estudos de caso** (de onde vêm os dados e como carregá-los). Não é preciso ler o código — é a "biblioteca" do workshop. Os dados e o material de apoio são baixados do repositório do workshop; sem internet, são usadas cópias locais da pasta `recursos/`.

In [ ]:
# ----------------------------------------------------------------
# Infraestrutura: fontes de dados e carregadores de cada caso
# ----------------------------------------------------------------
REPO = "https://raw.githubusercontent.com/DiogoCampanha/workshop-ia-responsavel/main/workshop_reduzido/recursos"
PASTAS_LOCAIS = ["recursos/dados", "dados"]   # cópias locais (uso offline)

# 1ª URL: repositório do workshop; 2ª: fonte original (reserva)
URLS = {
    "adult":   [f"{REPO}/dados/adult.csv",
                "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"],
    "compas":  [f"{REPO}/dados/compas.csv",
                "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"],
    "german":  [f"{REPO}/dados/german.csv",
                "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"],
    "law":     [f"{REPO}/dados/law.csv",
                "https://raw.githubusercontent.com/tailequy/fairness_dataset/main/experiments/data/law_school_clean.csv"],
    "student": [f"{REPO}/dados/student.csv",
                "https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-por.csv"],
}

def _ler(chave, **kw):
    """Tenta baixar o CSV; se falhar, usa a cópia local em recursos/dados/<chave>.csv"""
    for url in URLS[chave]:
        try:
            df = pd.read_csv(url, **kw)
            print(f"(dados baixados de {url.split('/')[2]})")
            return df
        except Exception:
            pass
    for pasta in PASTAS_LOCAIS:
        caminho = os.path.join(pasta, chave + ".csv")
        if os.path.exists(caminho):
            print(f"(sem internet — usando cópia local: {caminho})")
            return pd.read_csv(caminho, **kw)
    raise RuntimeError(f"Não foi possível carregar '{chave}'. Verifique a internet ou coloque o arquivo em recursos/dados/{chave}.csv")

# Material de apoio dos casos (mantido fora do notebook, no repositório do workshop)
URL_MATERIAL = f"{REPO}/material_instrutor.csv"

def _carregar_material():
    for origem in [URL_MATERIAL, "recursos/material_instrutor.csv", "material_instrutor.csv"]:
        try:
            m = pd.read_csv(origem)
            d = {r["dataset"]: dict(r) for _, r in m.iterrows()}
            for v in d.values():
                v["positivo_e_beneficio"] = str(v["positivo_e_beneficio"]).strip() == "True"
            return d
        except Exception:
            pass
    raise RuntimeError("Material de apoio do workshop não encontrado — avise o instrutor.")

MATERIAL = _carregar_material()

# ---------------- carregadores (um por caso) ----------------

def carregar_adult():
    nomes = ["idade","classe_trabalho","fnlwgt","educacao","educacao_anos","estado_civil",
             "ocupacao","relacao_familiar","raca","sexo","ganho_capital","perda_capital",
             "horas_semana","pais_origem","renda"]
    df = _ler("adult", header=None, names=nomes, na_values="?", skipinitialspace=True)
    df["alvo"] = (df["renda"].astype(str).str.contains(">50K")).astype(int)
    df = df.drop(columns=["renda","fnlwgt","educacao_anos"])
    return df

def carregar_compas():
    df = _ler("compas")
    df = df[(df["days_b_screening_arrest"] <= 30) & (df["days_b_screening_arrest"] >= -30)]
    df = df[(df["is_recid"] != -1) & (df["c_charge_degree"] != "O")]
    df = df[df["race"].isin(["African-American", "Caucasian"])]
    cols = ["sex","age","race","juv_fel_count","juv_misd_count","juv_other_count",
            "priors_count","c_charge_degree","two_year_recid"]
    df = df[cols].rename(columns={
        "sex":"sexo","age":"idade","race":"raca","juv_fel_count":"crimes_juvenis_graves",
        "juv_misd_count":"delitos_juvenis_leves","juv_other_count":"outros_reg_juvenis",
        "priors_count":"antecedentes","c_charge_degree":"grau_da_acusacao","two_year_recid":"alvo"})
    return df.reset_index(drop=True)

def carregar_german():
    nomes = ["conta_corrente","prazo_meses","historico_credito","proposito","valor_credito",
             "poupanca","tempo_emprego","perc_renda_comprometida","estado_civil_sexo",
             "outros_devedores","anos_na_residencia","patrimonio","idade","outros_financiamentos",
             "moradia","qtd_creditos_no_banco","tipo_emprego","qtd_dependentes","tem_telefone",
             "trabalhador_estrangeiro","risco"]
    df = _ler("german", sep=r"\s+", header=None, names=nomes)
    df["conta_corrente"] = df["conta_corrente"].map(
        {"A11":"saldo negativo","A12":"saldo baixo","A13":"saldo bom","A14":"sem conta"})
    df["historico_credito"] = df["historico_credito"].map(
        {"A30":"nenhum credito","A31":"pagou em dia","A32":"pagando em dia",
         "A33":"atrasos no passado","A34":"conta critica"})
    df["poupanca"] = df["poupanca"].map(
        {"A61":"<100","A62":"100-500","A63":"500-1000","A64":">1000","A65":"sem poupanca"})
    df["tempo_emprego"] = df["tempo_emprego"].map(
        {"A71":"desempregado","A72":"<1 ano","A73":"1-4 anos","A74":"4-7 anos","A75":">7 anos"})
    df["sexo"] = df["estado_civil_sexo"].map(
        {"A91":"masculino","A92":"feminino","A93":"masculino","A94":"masculino","A95":"feminino"})
    df = df.drop(columns=["estado_civil_sexo"])
    df["faixa_etaria"] = np.where(df["idade"] < 25, "jovem (<25)", "25 anos ou mais")
    df["alvo"] = (df["risco"] == 1).astype(int)   # 1 = bom pagador
    df = df.drop(columns=["risco"])
    return df

def carregar_law():
    df = _ler("law")
    df = df.rename(columns={"race":"raca","male":"masculino","lsat":"nota_LSAT","ugpa":"nota_graduacao",
                            "fam_inc":"renda_familiar","fulltime":"tempo_integral","tier":"nivel_faculdade",
                            "decile1b":"decil_1o_ano","decile3":"decil_3o_ano",
                            "zfygpa":"gpa_1o_ano_padronizado","zgpa":"gpa_final_padronizado"})
    df = df.dropna(subset=["pass_bar"])
    df["alvo"] = df["pass_bar"].astype(int)
    df = df.drop(columns=["pass_bar"])
    return df

def carregar_student():
    df = _ler("student", sep=";")
    df["alvo"] = (df["G3"] < 10).astype(int)   # 1 = em risco de reprovar
    # mantém só a nota do 1º período (G1): o alerta é dado no meio do ano, ANTES da nota final
    df = df.drop(columns=["G2","G3"]).rename(columns={"G1": "nota_1o_periodo"})
    df["zona"] = df["address"].map({"U":"urbana","R":"rural"})
    df = df.drop(columns=["address"])
    return df

# ---------------- registro dos casos ----------------
CONFIG = {
    "adult": carregar_adult, "compas": carregar_compas, "german": carregar_german,
    "law": carregar_law, "student": carregar_student,
}

def ficha(chave):
    c = MATERIAL[chave]
    print("=" * 78)
    print(c["titulo"])
    print("=" * 78)
    print(f"O modelo prevê ....... {c['preve']}")
    print(f"Previsão positiva .... {c['positivo']}")
    print("-" * 78)
    print("Contexto:", c["contexto"])
    print("=" * 78)

print("✅ Infraestrutura pronta. Casos disponíveis:", ", ".join(CONFIG.keys()))

---
# 🧪 PARTE 1 — Treinar um modelo, direto ao ponto (~15 min)

Um modelo de *machine learning* aprende padrões a partir de exemplos do passado, ajustando-se para **errar o mínimo possível *na média***. Guardem a palavra **média** — ela vai voltar.

## 1.1 · 🔧 DECISÃO 1 — Caso e modelo do grupo

Todos os casos usam dados **reais**, com problemas reais, e cada um esconde um viés e um dilema **diferentes**. **Descobrir onde mora o problema é tarefa de vocês.** 🕵️

| Caso | Tema | O modelo prevê... | | Modelo | Característica |
|---|---|---|---|---|---|
| `"adult"` | 💰 Renda (censo EUA) | quem ganha >US$ 50 mil/ano | | `"logistica"` | linear, simples |
| `"compas"` | ⚖️ Justiça criminal (EUA) | quem vai reincidir em 2 anos | | `"arvore"` | regras "se... então", interpretável |
| `"german"` | 🏦 Crédito (Alemanha) | quem é bom pagador | | `"floresta"` | várias árvores, mais precisa e menos interpretável |
| `"law"` | 🎓 Exame da ordem (EUA) | quem passa no *bar exam* | | | |
| `"student"` | 🏫 Escolas (Portugal) | quem está em risco de reprovar | | | |

In [ ]:
# 🔧 DECISÃO 1 — editem e executem
DATASET = "adult"      # "adult", "compas", "german", "law" ou "student"
MODELO  = "logistica"  # "logistica", "arvore" ou "floresta"

cfg = MATERIAL[DATASET]
df = CONFIG[DATASET]()
ficha(DATASET)
print(f"\n📦 {len(df)} linhas × {df.shape[1]-1} colunas. Cada linha é uma PESSOA real.")
print("Distribuição do alvo (o que queremos prever):")
print(df["alvo"].value_counts(normalize=True).rename({1:"  positivo (1)", 0:"  negativo (0)"}).round(3).to_string())
display(df.head(5))

## 1.2 · Treinar e avaliar

A célula abaixo faz tudo de uma vez: prepara os dados, separa **30% das pessoas para teste** (o modelo nunca as vê no treino), treina e avalia.

- Para ganhar tempo, **linhas com dados faltantes são removidas** (só afeta o caso `adult`, ~7% das linhas). Em projetos reais essa também é uma decisão ética: quem são as pessoas cujos dados "faltam"?
- Comparem a **acurácia** com o **chute da maioria** (prever sempre a classe mais comum) e olhem a **matriz de confusão**: falsos positivos (FP) e falsos negativos (FN) têm consequências bem diferentes para pessoas reais.

In [ ]:
# Preparação: remove linhas incompletas e transforma categorias em colunas 0/1
dados = df.dropna().copy()
X_bruto = dados.drop(columns=["alvo"])
X = pd.get_dummies(X_bruto, drop_first=False)

def novo_modelo(nome):
    return {
        "logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "arvore":    DecisionTreeClassifier(max_depth=6, random_state=SEED),
        "floresta":  RandomForestClassifier(n_estimators=120, random_state=SEED, n_jobs=-1),
    }[nome]

# separamos as PESSOAS (índices) em treino/teste — assim toda versão do modelo usa as mesmas pessoas
idx_tr, idx_te = train_test_split(dados.index, test_size=0.30, stratify=dados["alvo"], random_state=SEED)
X_tr, X_te = X.loc[idx_tr], X.loc[idx_te]
y_tr = dados.loc[idx_tr, "alvo"].values
y_te = dados.loc[idx_te, "alvo"].values

modelo = novo_modelo(MODELO).fit(X_tr, y_tr)
modelo_com, X_com, X_com_te = modelo, X, X_te          # guardado para a comparação da Parte 3

y_prev = modelo.predict(X_te)
acc = accuracy_score(y_te, y_prev)
chute = max(np.mean(y_te), 1 - np.mean(y_te))
print(f"✅ Modelo '{MODELO}' treinado com {len(X_tr)} pessoas e avaliado em {len(X_te)} pessoas nunca vistas.")
print(f"   (linhas removidas por dados faltantes: {len(df) - len(dados)})\n")
print(f"Acurácia do modelo: {acc:.1%}")
print(f"Chute da maioria:   {chute:.1%}   ← se o modelo não supera isso, ele não aprendeu nada útil")

mc = confusion_matrix(y_te, y_prev)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.imshow(mc, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{mc[i, j]:,}", ha="center", va="center",
                fontsize=15, color="black" if mc[i, j] < mc.max()*0.6 else "white")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["prev. negativo", "prev. positivo"])
ax.set_yticklabels(["real negativo", "real positivo"])
ax.set_title("Matriz de confusão (conjunto de teste)")
ax.set_xlabel("O que o modelo disse"); ax.set_ylabel("O que era verdade")
plt.tight_layout(); plt.show()

fp, fn = mc[0, 1], mc[1, 0]
print(f"⚠️ {fp} falsos positivos → {cfg['fp']}")
print(f"⚠️ {fn} falsos negativos → {cfg['fn']}")

### ✏️ TAREFA 1 — leitura rápida (2 min)
1. A acurácia superou o chute da maioria por quanto? Isso é muito ou pouco?
2. No **seu caso**, qual erro é pior: FP ou FN? Para **quem** ele é pior?

---
# 🕵️ PARTE 2 — Onde este modelo pode ser injusto? (~15 min)

## 2.1 · ✏️ A hipótese do grupo

O modelo está treinado e tem uma acurácia razoável. Agora, a pergunta que separa quem *usa* ML de quem *entende* ML:

> **Se este sistema entrasse em produção amanhã, quem poderia ser tratado injustamente por ele? Por quê?**

Só com o que vocês já viram (as colunas, a matriz de confusão, o contexto do caso), discutam:
1. Que **características das pessoas** poderiam levar o sistema a errar mais para umas do que para outras?
2. Quais colunas abaixo são **legítimas** para essa decisão? Quais são **eticamente suspeitas**? Alguma pode ser um **atalho disfarçado** para uma característica suspeita (ex.: CEP costuma "entregar" renda e raça)?
3. Registrem a **hipótese do grupo** na segunda célula abaixo — *antes* de ver a resposta.

In [ ]:
# As colunas que o SEU modelo está usando agora:
print(f"O modelo decide: {cfg['preve']}")
print(f"\nAs {X_bruto.shape[1]} colunas disponíveis para o modelo:\n")
for c in X_bruto.columns:
    exemplo = dados[c].dropna().unique()[:4]
    print(f"  • {c:<28} exemplos: {list(exemplo)}")

In [ ]:
# ✏️ Registrem a hipótese do grupo ANTES da revelação
HIPOTESES_DO_GRUPO = {
    "colunas_suspeitas":         [],   # ex.: ["idade", "..."] — colunas que NÃO deveriam pesar na decisão
    "quem_pode_ser_prejudicado": "",   # que tipo de pessoa este sistema pode tratar pior?
    "por_que":                   "",   # qual o mecanismo? (poucos dados? histórico injusto? atalhos?)
}
assert HIPOTESES_DO_GRUPO["quem_pode_ser_prejudicado"] != "", "⛔ Preencham a hipótese antes de seguir!"
print("Hipótese registrada ✅ — agora chamem o instrutor para receber o atributo sensível do caso.")

## 2.2 · 🎁 A revelação — o atributo sensível do seu caso

Cada caso tem um **atributo sensível**: uma característica protegida (por lei ou por ética) que **não deveria** determinar o resultado — e que historicamente determina. **O instrutor vai informar qual é o do seu grupo.** Preencham o nome exato da coluna e comparem com a hipótese.

In [ ]:
# ⬇️ preencham com o nome EXATO da coluna informada pelo instrutor (ex.: "sexo")
SENSIVEL = ""

assert SENSIVEL in dados.columns, f"⛔ Coluna '{SENSIVEL}' não existe. Colunas disponíveis: {list(dados.columns)}"
grupos_todos = dados[SENSIVEL].astype(str)     # o grupo de cada pessoa
gr_tr = grupos_todos.loc[idx_tr]
gr_te = grupos_todos.loc[idx_te]

print(f"Atributo sensível do caso: '{SENSIVEL}'")
print(f"Grupos: {sorted(grupos_todos.unique())}")
acertou = SENSIVEL in HIPOTESES_DO_GRUPO["colunas_suspeitas"]
print("\n🔎 O grupo tinha listado essa coluna como suspeita? " + ("SIM — boa intuição!" if acertou else "NÃO — discutam: por que ela passou despercebida?"))

## 2.3 · 🔧 DECISÃO 2 — Incluir ou excluir o atributo sensível?

A reação mais comum: *"então é só **remover** essa coluna que o modelo fica justo!"* Será? Essa estratégia tem nome — ***fairness through unawareness***. A célula abaixo treina **as duas versões** (com e sem a coluna). O grupo escolhe qual será o **modelo oficial**; na Parte 3 as duas serão comparadas automaticamente.

- `USAR_ATRIBUTO_SENSIVEL = True` → o modelo oficial **usa** a coluna (transparente, porém explícito).
- `USAR_ATRIBUTO_SENSIVEL = False` → o modelo oficial **não vê** a coluna ("cego").

*(A coluna continua guardada fora do modelo para auditoria — sem ela não conseguiríamos nem medir injustiça.)*

In [ ]:
# 🔧 DECISÃO 2 — editem e executem
USAR_ATRIBUTO_SENSIVEL = True        # True ou False

# versão SEM o atributo sensível (a versão COM foi treinada na Parte 1)
X_sem = pd.get_dummies(X_bruto.drop(columns=[SENSIVEL]), drop_first=False)
X_sem_tr, X_sem_te = X_sem.loc[idx_tr], X_sem.loc[idx_te]
modelo_sem = novo_modelo(MODELO).fit(X_sem_tr, y_tr)

acc_com = accuracy_score(y_te, modelo_com.predict(X_com_te))
acc_sem = accuracy_score(y_te, modelo_sem.predict(X_sem_te))
print(f"Acurácia COM '{SENSIVEL}': {acc_com:.2%}")
print(f"Acurácia SEM '{SENSIVEL}': {acc_sem:.2%}")
print(f"Diferença: {abs(acc_com-acc_sem):.2%} — " +
      ("quase nada muda. Suspeito, não? Se a coluna 'não fazia falta', que outras colunas carregam a mesma informação?"
       if abs(acc_com-acc_sem) < 0.01 else "houve mudança. O que o modelo perdeu (ou ganhou) ao ficar 'cego'?"))

# define o MODELO OFICIAL do grupo (usado nas Partes 3 a 5)
if USAR_ATRIBUTO_SENSIVEL:
    modelo, X, X_te = modelo_com, X_com, X_com_te
    print(f"\n➡️ Modelo oficial do grupo: COM a coluna '{SENSIVEL}'.")
else:
    modelo, X, X_te = modelo_sem, X_sem, X_sem_te
    print(f"\n➡️ Modelo oficial do grupo: SEM a coluna '{SENSIVEL}'.")
y_prev = modelo.predict(X_te)

---
# 🔍 PARTE 3 — Auditoria de *fairness* (~50 min)

Vamos auditar o modelo em ordem crescente de profundidade — dos dados às métricas de *fairness* — até esbarrar no conflito inevitável entre elas.

**Roteiro:** 3.1 representação → 3.2 desempenho por grupo → 3.3 taxa de seleção e regra dos 80% → 3.4 o que o modelo usa (e "cegar" resolve?) → 3.5 o conflito entre métricas.

> ⚠️ As Partes 1 e 2 precisam ter sido executadas (inclusive `SENSIVEL` e a Decisão 2).

## 3.1 · Representação — quem está nos dados?

**Viés de amostragem:** se um grupo é raro nos dados, o modelo aprende pouco sobre ele (e erra mais). **Viés histórico:** mesmo bem representado, um grupo pode carregar rótulos que refletem desigualdades do passado. A primeira auditoria é uma simples contagem.

In [ ]:
cont = grupos_todos.value_counts()
taxa_pos_real = dados.groupby(grupos_todos)["alvo"].mean().reindex(cont.index)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(cont.index.astype(str), cont.values, color="#4878CF")
axes[0].set_title(f"Quantas pessoas por grupo ({SENSIVEL})")
axes[0].set_ylabel("nº de pessoas")
axes[1].bar(taxa_pos_real.index.astype(str), taxa_pos_real.values, color="#EE854A")
axes[1].set_title("Taxa de alvo=1 REAL por grupo (dados históricos)")
axes[1].set_ylabel("proporção com alvo = 1")
for ax in axes: ax.tick_params(axis="x", rotation=10)
plt.tight_layout(); plt.show()

print(cont.to_string())
print()
for g, t in taxa_pos_real.items():
    print(f"Nos dados, {t:.1%} do grupo '{g}' tem alvo = 1")

### ✏️ TAREFA 2
1. Algum grupo é bem menor que o outro? Que consequência isso teve no aprendizado?
2. As taxas **reais** de alvo=1 já diferem entre grupos. Três explicações possíveis — qual(is) vale(m) para o seu caso?
   - (a) diferença real e legítima entre grupos;
   - (b) desigualdade histórica capturada nos dados;
   - (c) problema de **medição** do rótulo (o rótulo não mede o que a gente pensa que mede).
3. Se a diferença já vem dos dados, o modelo tem culpa de reproduzi-la? E quem a usa?

## 3.2 · Desempenho por subgrupo — o modelo erra igual para todos?

A acurácia global esconde desigualdades. Vamos abrir as métricas por grupo:

- **TPR** (*taxa de verdadeiros positivos*): dos que **são** alvo=1, quantos o modelo reconhece? TPRs iguais entre grupos = **igualdade de oportunidade** (*equal opportunity*).
- **FPR** (*taxa de falsos positivos*): dos que **não são** alvo=1, quantos o modelo marca errado? TPR **e** FPR iguais = ***equalized odds***.

*(Estes nomes são os conceitos formais da literatura — referência para aprofundar: Hardt et al., 2016.)*

In [ ]:
def metricas_por_grupo(y_true, y_pred, grupos):
    """Métricas de desempenho e fairness, calculadas grupo a grupo."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); grupos = np.asarray(grupos)
    linhas = []
    for g in sorted(pd.unique(grupos)):
        m = grupos == g
        yt, yp = y_true[m], y_pred[m]
        vp = int(((yt == 1) & (yp == 1)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
        vn = int(((yt == 0) & (yp == 0)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
        linhas.append({
            "grupo": g, "n": int(m.sum()),
            "acurácia":     (vp + vn) / max(len(yt), 1),
            "taxa_seleção": (vp + fp) / max(len(yt), 1),
            "TPR":          vp / max(vp + fn, 1),
            "FPR":          fp / max(fp + vn, 1),
            "precisão":     vp / max(vp + fp, 1),
        })
    return pd.DataFrame(linhas).set_index("grupo").round(3)

def disparate_impact(tabela):
    return tabela["taxa_seleção"].min() / max(tabela["taxa_seleção"].max(), 1e-9)

tabela = metricas_por_grupo(y_te, y_prev, gr_te)
display(tabela)

ax = tabela[["TPR", "FPR", "acurácia"]].plot.bar(figsize=(9, 3.8), rot=10,
        color=["#6ACC64", "#D65F5F", "#4878CF"])
ax.set_title("O modelo trata os grupos igual? (TPR, FPR e acurácia por grupo)")
ax.set_ylabel("taxa"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

print(f"Lembrete do seu caso — previsão positiva significa: {cfg['positivo']}")

## 3.3 · Taxa de seleção, *disparate impact* e a regra dos 80%

**Taxa de seleção** = % do grupo que recebe previsão **positiva**. A razão entre a menor e a maior taxa é o ***disparate impact ratio***. Na legislação trabalhista dos EUA, uma razão **abaixo de 0,8** ("regra dos 80%", EEOC 1978) é indício de discriminação — taxas de seleção iguais correspondem ao conceito de **paridade demográfica**.

> No seu caso a previsão positiva é benefício ou dano? Isso muda completamente a leitura da barra abaixo.

In [ ]:
sel = tabela["taxa_seleção"]
razao = disparate_impact(tabela)

ax = sel.plot.bar(color="#956CB4", rot=10, figsize=(7.5, 3.5))
ax.set_title("Taxa de seleção por grupo (% que recebe previsão POSITIVA)")
ax.set_ylabel("taxa de seleção")
plt.tight_layout(); plt.show()

print(f"Disparate impact ratio (menor/maior): {razao:.2f}")
print("Regra dos 80%: " + ("✅ PASSA (razão ≥ 0,80)" if razao >= 0.8 else "❌ VIOLADA (razão < 0,80) — indício de impacto desproporcional"))
print(f"\nInterpretação no seu caso: previsão positiva = {cfg['positivo']}")
if not cfg["positivo_e_beneficio"]:
    print("🚨 ATENÇÃO: neste caso a previsão positiva PREJUDICA a pessoa — grupo com MAIOR taxa de seleção é o mais penalizado!")

## 3.4 · O que o modelo usa? E "cegar" o modelo resolve?

Se o modelo se apoia no atributo sensível — ou em variáveis fortemente correlacionadas a ele (***proxies***: CEP ↔ raça, altura ↔ sexo...) — a discriminação entra mesmo "sem querer". A célula abaixo faz três coisas:

1. mostra as variáveis mais usadas pelo modelo oficial;
2. compara **automaticamente** as métricas por grupo do modelo **COM** e **SEM** o atributo sensível (as duas versões treinadas na Decisão 2);
3. testa se as **outras** colunas conseguem "adivinhar" o grupo de cada pessoa (um detector de *proxies*).

*(Para ir além: **LIME** e **SHAP** explicam previsões individuais.)*

In [ ]:
# (1) Em que o modelo oficial se apoia?
if MODELO == "logistica":
    imp = pd.Series(np.abs(modelo[-1].coef_[0]), index=X.columns)
    tipo = "peso absoluto (|coeficiente| padronizado)"
else:
    imp = pd.Series(modelo.feature_importances_, index=X.columns)
    tipo = "importância (redução de impureza)"
top = imp.sort_values(ascending=False).head(12)
cores = ["#D65F5F" if c.startswith(SENSIVEL) else "#4878CF" for c in top.index]
ax = top.iloc[::-1].plot.barh(color=cores[::-1], figsize=(8.5, 4.5))
ax.set_title(f"12 variáveis mais usadas pelo modelo oficial — {tipo}")
plt.tight_layout(); plt.show()

# (2) "Cegar" o modelo resolve? Comparação automática COM × SEM o atributo sensível
cols = ["taxa_seleção", "TPR", "FPR", "acurácia"]
t_com = metricas_por_grupo(y_te, modelo_com.predict(X_com_te), gr_te)
t_sem = metricas_por_grupo(y_te, modelo_sem.predict(X_sem_te), gr_te)
print(f"Métricas por grupo — modelo COM × SEM a coluna '{SENSIVEL}':")
display(pd.concat({f"COM '{SENSIVEL}'": t_com[cols], f"SEM '{SENSIVEL}'": t_sem[cols]}, axis=1))
print(f"Disparate impact: COM = {disparate_impact(t_com):.2f} | SEM = {disparate_impact(t_sem):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))
for ax, met in zip(axes, ["taxa_seleção", "TPR", "FPR"]):
    pd.DataFrame({"COM atributo": t_com[met], "SEM atributo": t_sem[met]}).plot.bar(
        ax=ax, rot=10, color=["#4878CF", "#B0B0B0"])
    ax.set_title(met)
fig.suptitle(f"Remover '{SENSIVEL}' muda o tratamento dos grupos?")
plt.tight_layout(); plt.show()

# (3) Detector de proxies: as OUTRAS colunas conseguem adivinhar o grupo?
detector = DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(X_sem_tr, gr_tr)
acerto_det = accuracy_score(gr_te, detector.predict(X_sem_te))
chute_det = gr_te.value_counts(normalize=True).max()
pistas = pd.Series(detector.feature_importances_, index=X_sem.columns).sort_values(ascending=False).head(3)
print(f"\n🕵️ Detector de proxies: SEM ver '{SENSIVEL}', um modelo simples acerta o grupo de {acerto_det:.0%} das pessoas")
print(f"   (chutando sempre o grupo maior, acertaria {chute_det:.0%}). Colunas que mais 'entregam' o grupo: {list(pistas.index)}")

### ✏️ TAREFA 3 — "cegar" o modelo resolve?
1. Comparem COM × SEM: as taxas de seleção, TPR e FPR por grupo mudaram muito?
2. O detector de *proxies* adivinhou o grupo com que frequência? As colunas que "entregam" o grupo fazem sentido no seu caso?
3. Na maioria dos casos, **quase nada muda** — outras variáveis carregam a mesma informação (codificação redundante). O que isso diz sobre a estratégia de simplesmente apagar a coluna sensível?

## 3.5 · 💥 O conflito entre métricas — ajustando o limiar de decisão

**Ponto central do workshop.** O modelo não decide "sim/não": ele gera uma **probabilidade**, e nós escolhemos o **limiar** de corte (padrão 0,5). Podemos usar **limiares diferentes por grupo** para "consertar" uma injustiça — essa é a mitigação por *pós-processamento* (Hardt et al., 2016).

**Sua missão:** edite os limiares e tente **igualar a taxa de seleção** entre os grupos (paridade demográfica). Depois observe o que aconteceu com TPR, FPR e precisão. Em seguida tente igualar o TPR. Consegue igualar tudo ao mesmo tempo?

In [ ]:
# Probabilidades previstas para o conjunto de teste (base de tudo nesta seção)
probas = modelo.predict_proba(X_te)[:, 1]
NOMES_GRUPOS = sorted(pd.unique(np.asarray(gr_te)))

def prever_com_limiares(probas, grupos, limiares):
    grupos = np.asarray(grupos)
    yp = np.zeros(len(probas), dtype=int)
    for g, t in limiares.items():
        m = grupos == g
        yp[m] = (probas[m] >= t).astype(int)
    return yp

def painel(limiares):
    base = metricas_por_grupo(y_te, (probas >= 0.5).astype(int), gr_te)
    novo = metricas_por_grupo(y_te, prever_com_limiares(probas, gr_te, limiares), gr_te)
    comp = pd.concat({"limiar único 0,50": base, "seus limiares": novo}, axis=1)
    display(comp.round(3))
    for nome, t in [("limiar único 0,50", base), ("seus limiares", novo)]:
        r = disparate_impact(t)
        acc_g = accuracy_score(y_te, (probas >= 0.5).astype(int) if "único" in nome
                               else prever_com_limiares(probas, gr_te, limiares))
        print(f"{nome:>18}: acurácia global {acc_g:.1%} | disparate impact {r:.2f} "
              + ("✅80%" if r >= 0.8 else "❌80%"))

# Como cada métrica reage ao limiar, grupo a grupo:
ts = np.linspace(0.05, 0.95, 46)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for g in NOMES_GRUPOS:
    m = np.asarray(gr_te) == g
    sel_c, tpr_c, fpr_c = [], [], []
    for t in ts:
        yp = (probas[m] >= t).astype(int); yt = y_te[m]
        sel_c.append(yp.mean())
        tpr_c.append(((yt == 1) & (yp == 1)).sum() / max((yt == 1).sum(), 1))
        fpr_c.append(((yt == 0) & (yp == 1)).sum() / max((yt == 0).sum(), 1))
    axes[0].plot(ts, sel_c, label=g); axes[1].plot(ts, tpr_c, label=g); axes[2].plot(ts, fpr_c, label=g)
for ax, titulo in zip(axes, ["taxa de seleção", "TPR", "FPR"]):
    ax.set_title(titulo + " × limiar"); ax.set_xlabel("limiar"); ax.axvline(0.5, ls=":", c="gray")
axes[0].legend(title=SENSIVEL)
plt.tight_layout(); plt.show()
print("👆 Use estas curvas como 'mapa': onde cortar cada grupo para igualar a métrica que você quer?")

In [ ]:
# 🔧 DECISÃO 3 — editem os limiares por grupo e executem (quantas vezes quiserem!)
LIMIARES = {g: 0.50 for g in NOMES_GRUPOS}
# Exemplo: LIMIARES = {"Female": 0.35, "Male": 0.55}  (usem os nomes exatos impressos abaixo)
print("Grupos:", NOMES_GRUPOS)

painel(LIMIARES)

### ✏️ TAREFA 4 — o que vocês acabaram de descobrir
1. Ao igualar a **taxa de seleção**, o que aconteceu com o TPR e o FPR de cada grupo? E com a acurácia global?
2. Ao igualar o **TPR**, a taxa de seleção se igualou? E a precisão?
3. Deixem na célula acima os limiares "finais" do grupo — eles vão para o painel final (Parte 5).

### 🧠 Por que é impossível agradar todas as métricas?
Quando as **taxas-base** (a % real de alvo=1) diferem entre grupos, é **matematicamente impossível** igualar simultaneamente calibração, TPR e FPR — é o **teorema da impossibilidade do fairness** (Kleinberg et al., 2016; Chouldechova, 2016). Não é limitação do seu modelo nem falta de dados: **é aritmética**.

**Consequência:** "ser justo" exige **escolher** qual definição de justiça priorizar — e essa escolha é **ética e contextual**, não técnica. Quem deve fazê-la? O dev? A empresa? A lei? As pessoas afetadas?

---
# 🧩 PARTE 4 — O dilema ético do seu caso (~10 min)

Cada caso deste workshop esconde um dilema **diferente**. Executem a célula abaixo para revelar o do seu grupo — e decidam juntos qual opção defender.

In [ ]:
print(MATERIAL[DATASET]["dilema"])

In [ ]:
# 🔧 DECISÃO 4 — registrem a posição do grupo (vai para o painel final)
DECISAO_DO_GRUPO = {
    "opcao_escolhida":         "",   # "A", "B", "C" (ou uma proposta própria do grupo)
    "quem_ganha_e_quem_perde": "",   # todo trade-off tem um lado que perde — qual é?
    "justificativa":           "",   # por que essa opção? que valores pesaram?
}
print("Decisão registrada ✅")

---
# 🎤 PARTE 5 — Painel final e apresentação relâmpago (~15 min)

A célula abaixo resume tudo o que o grupo fez em **uma tela**. Ela é a base da apresentação relâmpago (**~2,5 min por grupo**), que responde a três perguntas:

1. **Que viés encontramos — e onde ele nasce?** (dados, rótulo, variáveis, uso?)
2. **O que a nossa correção melhorou — e o que piorou?** (use as diferenças entre grupos do painel)
3. **Que posição tomamos no dilema — e quem paga o preço?**

In [ ]:
def painel_final():
    antes  = metricas_por_grupo(y_te, (probas >= 0.5).astype(int), gr_te)
    depois = metricas_por_grupo(y_te, prever_com_limiares(probas, gr_te, LIMIARES), gr_te)
    acc_a = accuracy_score(y_te, (probas >= .5).astype(int))
    acc_d = accuracy_score(y_te, prever_com_limiares(probas, gr_te, LIMIARES))
    print("=" * 78)
    print("PAINEL FINAL — " + cfg["titulo"])
    print("=" * 78)
    print(f"Modelo: {MODELO} | atributo sensível: '{SENSIVEL}' | usado no modelo: {USAR_ATRIBUTO_SENSIVEL}")
    print(f"Limiares por grupo: {LIMIARES}")
    print("\n1) O VIÉS — " + cfg["tipo_de_vies"])
    print("\n2) A CORREÇÃO — diferença entre o melhor e o pior grupo em cada métrica:")
    for met in ["taxa_seleção", "TPR", "FPR", "precisão"]:
        ga, gd = antes[met].max() - antes[met].min(), depois[met].max() - depois[met].min()
        seta = "→ igual" if abs(gd - ga) < 0.005 else ("↓ diminuiu (mais equilibrado)" if gd < ga else "↑ aumentou (menos equilibrado)")
        print(f"   {met:<13} {ga:.3f} → {gd:.3f}   {seta}")
    print(f"   disparate impact  {disparate_impact(antes):.2f} → {disparate_impact(depois):.2f}"
          f"   (regra dos 80%: {'✅' if disparate_impact(depois) >= .8 else '❌'})")
    print(f"   acurácia global   {acc_a:.1%} → {acc_d:.1%}")
    print("\n3) A DECISÃO DO GRUPO:")
    for k, v in DECISAO_DO_GRUPO.items():
        print(f"   {k}: {v or '(preencher na Decisão 4)'}")
    print("=" * 78)

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
    for ax, met in zip(axes, ["taxa_seleção", "TPR", "FPR"]):
        pd.DataFrame({"limiar único": antes[met], "com correção": depois[met]}).plot.bar(
            ax=ax, rot=10, color=["#4878CF", "#6ACC64"])
        ax.set_title(met)
    fig.suptitle("Antes × depois da correção — figura para a apresentação")
    plt.tight_layout(); plt.show()

painel_final()

## Para levar para a vida

- Acurácia global esconde injustiças — **desagregue por grupo, sempre**.
- Remover a variável sensível **não** resolve (*proxies*!). E medir exige *ter* a variável.
- Não existe "justo" universal: existe a **escolha** de qual justiça priorizar — ética e contextual.
- Decisões técnicas "neutras" (limpar dados, escolher métrica, cortar limiar) **moldam vidas**.

📚 Referências, fontes dos dados e a versão estendida do workshop: [github.com/DiogoCampanha/workshop-ia-responsavel](https://github.com/DiogoCampanha/workshop-ia-responsavel)

🎉 **Obrigado por participarem!**